Python Code Sample

This notebook demonstrates, end to end:

1. **Data cleaning** — normalizing messy person-level records (accents, nicknames,
   misspellings, name suffixes, city spelling variants, invalid geography), with the
   attrition of every filter reported;
2. **Record linkage** — a blocked deterministic **matching ladder** with sequential
   lock-in, timing tests, and age disambiguation;
3. **Validation** — because the data are synthetic with known ground truth, the
   matcher's per-step precision, recall, and false-match rate are *measured*, not assumed;
4. **Panel assembly** — merging the matched sources into an inventor–year panel;
5. **Figures & tables** — a matching funnel, coverage over time, and a two-way
   fixed-effects **event study** whose estimate is checked against the planted effect.

Runs end to end in well under a minute. Requires only
`pip install polars numpy rapidfuzz matplotlib`.

In [ ]:
import os, time, warnings
import numpy as np
import polars as pl

# silence a polars 2.0 forward-compat notice on list explode (no empty lists here)
warnings.filterwarnings("ignore", message=".*empty_as_null.*")

t00 = time.time()
rng = np.random.default_rng(20260804)

DATA, OUT = "sample_data", "output"
os.makedirs(DATA, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

YEARS = (2006, 2025)     # observation window (filing years, directory years)
TIMING_SLACK = 2         # a directory presence year within +/-2 of a filing year
DL_MAX = 2               # Damerau-Levenshtein tolerance for misspelled first names
AGE_TOL = 2              # |directory birth year - inferred birth year| tolerance
TRUE_ATT = 0.15          # planted effect: +15pp homeownership after first filing
EVENT_WIN = (-5, 5)      # event-study window around the first patent filing

## 1 · Synthetic raw data with known truth

Two sources, mimicking the structure (and the pathologies) of the real ones:

- **Patent-side inventor records** — one row per inventor × patent, with *free-text*
  name and location fields as filed: nicknames (`WILLIAM` → `Bill`), typos
  (transpositions, dropped/doubled letters), accents (`JOSÉ`), suffixes (`Smith Jr.`),
  inconsistent middle initials, city variants (`St. Louis` / `SAINT LOUIS`), and a few
  foreign or empty geography codes. The same inventor's spelling can differ across filings.
- **Directory person–years** — a household-directory panel (one row per person × year,
  as in Data Axle): canonical names, presence spells with gaps, reported age (with
  occasional error and missingness), and a homeownership flag.

The homeownership process plants a **+15 pp jump at the inventor's first filing year**,
on top of a linear age profile and a person effect — Section 7's event study must
recover it. A held-out set of inventors is **absent from the directory entirely**, so
the false-match rate is measurable. `inventor_truth.parquet` (inventor → true person)
is used **only for grading**, never by the matcher.

In [ ]:
FIRST_NICK = {   # canonical -> nickname pool (directory stores the canonical form)
    "WILLIAM": ["BILL", "WILL"], "ROBERT": ["BOB", "ROB"], "MICHAEL": ["MIKE"],
    "ELIZABETH": ["LIZ", "BETH"], "KATHERINE": ["KATE", "KATHY"],
    "RICHARD": ["RICK", "DICK"], "THOMAS": ["TOM"], "CHRISTOPHER": ["CHRIS"],
    "JENNIFER": ["JEN"], "MARGARET": ["MEG", "PEGGY"], "JAMES": ["JIM"],
    "JOSEPH": ["JOE"], "DANIEL": ["DAN"], "ANDREW": ["ANDY"], "MATTHEW": ["MATT"],
    "NICHOLAS": ["NICK"], "ANTHONY": ["TONY"], "STEVEN": ["STEVE"],
    "EDWARD": ["ED", "TED"], "BENJAMIN": ["BEN"], "SAMUEL": ["SAM"],
    "PATRICIA": ["PAT", "TRICIA"],
}
FIRST_ACCENT = {"JOSE": "José", "ANDRE": "André", "RENEE": "Renée",
                "FRANCOIS": "François", "BJORN": "Björn", "ZOE": "Zoë"}
FIRST_PLAIN = [
    "OLIVER", "SOPHIA", "EMMA", "LUCAS", "AMELIA", "HENRY", "GRACE", "WEI", "LI",
    "MEI", "JING", "CHENG", "RAVI", "PRIYA", "ARJUN", "ANANYA", "CARLOS", "MARIA",
    "DIEGO", "LUCIA", "AHMED", "FATIMA", "OMAR", "LEILA", "HIROSHI", "YUKI",
    "KENJI", "SOOJIN", "MINJUN", "JUAN", "ANA", "IVAN", "OLGA", "LARS", "INGRID",
    "MARCO", "GIULIA", "PIERRE", "CLAIRE", "SEBASTIAN", "NORA", "DMITRI", "ELENA",
    "HANNAH", "ETHAN", "CHLOE", "AIDEN", "MAYA", "FELIX", "IRIS",
]
FIRSTS = sorted(set(FIRST_NICK) | set(FIRST_ACCENT) | set(FIRST_PLAIN))

LASTS = sorted(set("""SMITH JOHNSON WILLIAMS BROWN JONES GARCIA MILLER DAVIS
RODRIGUEZ MARTINEZ HERNANDEZ LOPEZ GONZALEZ WILSON ANDERSON THOMAS TAYLOR MOORE
JACKSON MARTIN LEE PEREZ THOMPSON WHITE HARRIS SANCHEZ CLARK RAMIREZ LEWIS
ROBINSON WALKER YOUNG ALLEN KING WRIGHT SCOTT TORRES NGUYEN HILL FLORES GREEN
ADAMS NELSON BAKER HALL RIVERA CAMPBELL MITCHELL CARTER ROBERTS GOMEZ PHILLIPS
EVANS TURNER DIAZ PARKER CRUZ EDWARDS COLLINS REYES STEWART MORRIS MORALES MURPHY
COOK ROGERS GUTIERREZ ORTIZ MORGAN COOPER PETERSON BAILEY REED KELLY HOWARD RAMOS
KIM CHO PARK CHEN ZHANG WANG LIU ZHAO HUANG XU LIN PATEL SHAH SHARMA GUPTA SINGH
KUMAR REDDY IYER TANAKA SATO SUZUKI TAKAHASHI WATANABE MULLER SCHMIDT SCHNEIDER
FISCHER WEBER MEYER WAGNER BECKER HOFFMANN ROSSI RUSSO FERRARI ESPOSITO BIANCHI
DUBOIS MOREAU LAURENT SIMON MICHEL LEROY IVANOV PETROV SOKOLOV POPOV KOWALSKI
NOWAK SILVA SANTOS OLIVEIRA PEREIRA COSTA ALMEIDA HANSEN JENSEN NILSSON BERG
LINDBERG OCONNOR MCDONALD MCCARTHY OBRIEN GALLAGHER FITZGERALD VANDERBERG
DEVRIES JANSSEN BAKKER VISSER""".split()))

CITIES = {   # state -> canonical city list (the directory stores these forms)
    "CA": ["SAN JOSE", "PALO ALTO", "MOUNTAIN VIEW", "SAN DIEGO", "FREMONT", "IRVINE"],
    "WA": ["SEATTLE", "REDMOND", "BELLEVUE", "KIRKLAND"],
    "TX": ["AUSTIN", "HOUSTON", "FORT WORTH", "PLANO", "DALLAS"],
    "MA": ["CAMBRIDGE", "BOSTON", "WALTHAM", "SOMERVILLE"],
    "NY": ["NEW YORK", "YONKERS", "MOUNT VERNON", "ALBANY", "ITHACA"],
    "MO": ["SAINT LOUIS", "KANSAS CITY", "COLUMBIA"],
    "MN": ["SAINT PAUL", "MINNEAPOLIS", "ROCHESTER"],
    "NC": ["DURHAM", "RALEIGH", "CHARLOTTE", "CARY"],
}
CITY_MESSY = {   # canonical -> raw variants seen on the patent side
    "SAINT LOUIS": ["St. Louis", "ST LOUIS", "Saint Louis"],
    "SAINT PAUL": ["St. Paul", "ST PAUL"],
    "MOUNT VERNON": ["Mt. Vernon", "MT VERNON"],
    "FORT WORTH": ["Ft. Worth", "FT WORTH"],
}
STATES = list(CITIES)
state_w = np.array([len(CITIES[s]) for s in STATES], float)
state_w /= state_w.sum()

# ---- person universe --------------------------------------------------------
N_PERSONS = 40_000
MI_POOL = list("ABCDEFGHJKLMNPRSTW")

state = rng.choice(STATES, N_PERSONS, p=state_w)
city = np.empty(N_PERSONS, object)
for s in STATES:
    idx = np.flatnonzero(state == s)
    city[idx] = rng.choice(CITIES[s], idx.size)
mi_mask = rng.random(N_PERSONS) < 0.72
mi_draw = rng.choice(MI_POOL, N_PERSONS)
persons = pl.DataFrame({
    "person_id": np.arange(N_PERSONS),
    "state": state, "city": city,
    "first_name": rng.choice(FIRSTS, N_PERSONS),
    "last_name": rng.choice(LASTS, N_PERSONS),
    "middle_initial": [m if k else None for m, k in zip(mi_draw, mi_mask)],
    "gender": rng.choice(["M", "F"], N_PERSONS),
    "birth_year": rng.integers(1945, 1996, N_PERSONS),
    "span_start": rng.integers(YEARS[0], 2013, N_PERSONS),
    "pers_eff": np.clip(rng.normal(0.0, 0.07, N_PERSONS), -0.15, 0.15),
})
persons = persons.with_columns(
    span_end=pl.min_horizontal(pl.col("span_start")
                               + pl.Series(rng.integers(8, 21, N_PERSONS)),
                               pl.lit(YEARS[1])))

# ---- inventors: 6,000 in the directory + 400 deliberately absent ------------
N_INV_DIR, N_INV_OUT = 6_000, 400
perm = rng.permutation(N_PERSONS)
inv_pid = perm[:N_INV_DIR]
print(f"person universe: {N_PERSONS:,} | directory inventors: {N_INV_DIR:,} "
      f"| out-of-directory inventors: {N_INV_OUT:,}")

In [ ]:
# ---- distractor twins: same (state, city, last, first) block as an inventor -
# "resolvable" twins differ in birth year (>=8y) and live on their own spell ->
# the timing or age steps can separate them. "hard" twins share the spell and
# are within +/-1 birth year -> designed to stay ambiguous.
n_res, n_hard = int(N_INV_DIR * 0.10), int(N_INV_DIR * 0.03)
res_src = inv_pid[:n_res]
hard_src = inv_pid[n_res:n_res + n_hard]

src = persons.filter(pl.col("person_id").is_in(res_src))
res_twins = src.with_columns(
    person_id=pl.Series(np.arange(len(res_src)) + N_PERSONS),
    middle_initial=pl.Series([rng.choice(MI_POOL) if rng.random() < 0.6 else None
                              for _ in range(len(res_src))]),
    birth_year=pl.col("birth_year")
        + pl.Series(rng.integers(8, 19, len(res_src)) * rng.choice([-1, 1], len(res_src))),
    span_start=pl.Series(rng.integers(YEARS[0], 2013, len(res_src))),
    pers_eff=pl.Series(np.clip(rng.normal(0.0, 0.07, len(res_src)), -0.15, 0.15)),
).with_columns(
    span_end=pl.min_horizontal(pl.col("span_start")
                               + pl.Series(rng.integers(8, 21, len(res_src))),
                               pl.lit(YEARS[1])))

src = persons.filter(pl.col("person_id").is_in(hard_src))
hard_twins = src.with_columns(   # same spell, birth within +/-1: unresolvable
    person_id=pl.Series(np.arange(len(hard_src)) + N_PERSONS + n_res),
    birth_year=pl.col("birth_year") + pl.Series(rng.integers(-1, 2, len(hard_src))),
    pers_eff=pl.Series(np.clip(rng.normal(0.0, 0.07, len(hard_src)), -0.15, 0.15)),
)
persons = pl.concat([persons, res_twins, hard_twins])
persons = persons.with_columns(birth_year=pl.col("birth_year").clip(1940, 2000))
print(f"universe incl. twins: {persons.height:,} "
      f"(resolvable {n_res:,}, hard {n_hard:,})")

In [ ]:
# ---- patents: 1-6 filings per inventor, inside the directory spell ----------
pp = persons.filter(pl.col("person_id").is_in(inv_pid)).select(
    "person_id", "state", "city", "first_name", "last_name", "middle_initial",
    "gender", "birth_year", "span_start", "span_end")
rows = pp.to_dicts()
inv_defs = []
for i, r in enumerate(rows):
    lo, hi = max(r["span_start"], 2008), min(r["span_end"], 2023)
    fy = sorted(rng.integers(lo, hi + 1, rng.integers(1, 7)).tolist())
    inv_defs.append({**r, "inventor_id": f"INV-{i:05d}", "filing_years": fy})
for j in range(N_INV_OUT):   # inventors with no directory record at all
    st = str(rng.choice(STATES))
    inv_defs.append({
        "person_id": None, "state": st, "city": str(rng.choice(CITIES[st])),
        "first_name": str(rng.choice(FIRSTS)), "last_name": str(rng.choice(LASTS)),
        "middle_initial": str(rng.choice(MI_POOL)) if rng.random() < 0.6 else None,
        "gender": str(rng.choice(["M", "F"])),
        "birth_year": int(rng.integers(1945, 1996)),
        "span_start": None, "span_end": None,
        "inventor_id": f"INV-{N_INV_DIR + j:05d}",
        "filing_years": sorted(rng.integers(2008, 2024, rng.integers(1, 7)).tolist()),
    })
first_year = {d["inventor_id"]: d["filing_years"][0] for d in inv_defs}
truth = pl.DataFrame({"inventor_id": [d["inventor_id"] for d in inv_defs],
                      "true_person_id": [d["person_id"] for d in inv_defs]})
truth.write_parquet(f"{DATA}/inventor_truth.parquet")

# ---- directory person-years: spells with gaps, age noise, ownership DGP -----
inv_first = pl.DataFrame(
    [(d["person_id"], d["filing_years"][0]) for d in inv_defs
     if d["person_id"] is not None],
    schema=["person_id", "first_filing"], orient="row")
py = (persons.join(inv_first, on="person_id", how="left")
      .with_columns(year=pl.int_ranges("span_start", pl.col("span_end") + 1))
      .explode("year"))
py = py.filter(pl.Series(rng.random(py.height)) < 0.92)          # presence gaps
age_err = rng.choice([-1, 0, 1], py.height, p=[0.03, 0.94, 0.03])
age_na = rng.random(py.height) < 0.05
post = (pl.col("year") >= pl.col("first_filing")).fill_null(False)
py = (py.with_columns(
        age=pl.col("year") - pl.col("birth_year") + pl.Series(age_err),
        post=post)
      .with_columns(p_own=(0.30 + 0.011 * (pl.col("age") - 30)
                           + pl.col("pers_eff")
                           + TRUE_ATT * pl.col("post").cast(pl.Float64))
                    .clip(0.03, 0.97))
      .with_columns(homeowner=pl.Series(rng.random(py.height)) < pl.col("p_own"),
                    age=pl.when(pl.Series(age_na)).then(None).otherwise(pl.col("age"))))
directory = py.select("person_id", "year", "state", "city", "first_name",
                      "last_name", "middle_initial", "gender", "age", "homeowner")
directory.write_parquet(f"{DATA}/directory_person_years.parquet")
print(f"directory person-years: {directory.height:,} "
      f"({directory['person_id'].n_unique():,} persons, "
      f"{YEARS[0]}-{YEARS[1]}, presence rate 92%)")

In [ ]:
# ---- patent-side records: one messy row per inventor x patent ---------------
def typo(s):
    if len(s) < 4:
        return s
    i = int(rng.integers(1, len(s) - 2))
    op = rng.integers(3)
    if op == 0:
        return s[:i] + s[i + 1] + s[i] + s[i + 2:]   # transposition
    if op == 1:
        return s[:i] + s[i + 1:]                     # deletion
    return s[:i] + s[i] + s[i:]                      # doubled letter

recs, pid = [], 0
for d in inv_defs:
    for fy in d["filing_years"]:
        first = d["first_name"]
        if first in FIRST_ACCENT and rng.random() < 0.60:
            first = FIRST_ACCENT[first]              # José, René e...
        elif first in FIRST_NICK and rng.random() < 0.15:
            first = str(rng.choice(FIRST_NICK[first]))   # Bill for William
        if rng.random() < 0.06:
            first = typo(first)
        first = first.title() if rng.random() < 0.85 else first.upper()
        if d["middle_initial"] and rng.random() < 0.60:
            first = f"{first} {d['middle_initial']}."     # "Daniel R."
        if rng.random() < 0.02:
            first = ""                                    # unusable record
        last = d["last_name"]
        if rng.random() < 0.04:
            last = typo(last)
        last = last.title()
        if rng.random() < 0.05:
            last += str(rng.choice([" Jr.", " Sr.", " III"]))
        cty = d["city"]
        cty = (str(rng.choice(CITY_MESSY[cty])) if cty in CITY_MESSY
               and rng.random() < 0.55 else cty.title())
        st = d["state"] if rng.random() > 0.015 else str(rng.choice(["ON", "BC", "XX"]))
        recs.append({"patent_id": f"P{pid:06d}", "inventor_id": d["inventor_id"],
                     "filing_year": fy, "raw_first": first, "raw_last": last,
                     "raw_city": cty, "raw_state": st})
        pid += 1
records = pl.DataFrame(recs)
records.write_parquet(f"{DATA}/patent_inventor_records.parquet")

# birth year inferred from education histories (as in the production pipeline's
# LinkedIn arc): available for ~55% of inventors, +/-1y noise, 10% outliers
est = []
for d in inv_defs:
    if rng.random() < 0.55:
        by = d["birth_year"] + (int(rng.integers(3, 6)) * int(rng.choice([-1, 1]))
                                if rng.random() < 0.10 else int(rng.integers(-1, 2)))
        est.append({"inventor_id": d["inventor_id"], "birth_year_est": by})
pl.DataFrame(est).write_parquet(f"{DATA}/inventor_birth_est.parquet")

print(f"patent-side records: {records.height:,} rows, "
      f"{records['inventor_id'].n_unique():,} inventors")
print("\na taste of the mess going in:")
print(records.filter(pl.col("raw_city").str.contains(r"\.")
                     | pl.col("raw_last").str.contains(r"\."))
      .head(6).select("raw_first", "raw_last", "raw_city", "raw_state"))

## 2 · Cleaning the patent-side records

Normalization (adapted from the production pipeline's shared module): NFKD-fold
accents to ASCII, uppercase, strip punctuation, collapse whitespace, expand
`ST/MT/FT` city prefixes, strip generational suffixes from surnames, then split the
free-text first-name field into **first / middle-initial** and derive the
3-character blocking prefix `first3`.

Every filter reports its attrition in both rows and unique inventors — the funnel
convention of Bernstein et al. (2026, Table A.1) that the production pipeline
prints at every stage. Records are dropped only for *unusability* (no name, no
valid U.S. geography), never for inconvenience.

In [ ]:
US_STATES = set("""AL AK AZ AR CA CO CT DE FL GA HI ID IL IN IA KS KY LA ME MD
MA MI MN MS MO MT NE NV NH NJ NM NY NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA
WV WI WY DC PR""".split())
NAME_SUFFIXES = ["JR", "SR", "II", "III", "IV"]

def norm_text(col):
    """Uppercase ASCII fold; punctuation -> space; collapse; '' -> null."""
    return (col.str.normalize("NFKD").str.replace_all(r"\p{M}", "")
            .str.to_uppercase().str.replace_all(r"[-/.]", " ")
            .str.replace_all(r"[^A-Z0-9 ]", "").str.replace_all(r"\s+", " ")
            .str.strip_chars().replace("", None))

def norm_city(col):
    return (norm_text(col).str.replace(r"^ST ", "SAINT ")
            .str.replace(r"^MT ", "MOUNT ").str.replace(r"^FT ", "FORT "))

def strip_suffix(col):
    pat = r" (?:" + "|".join(NAME_SUFFIXES) + r")$"
    return col.str.replace(pat, "").str.strip_chars().replace("", None)

step = [("raw inventor-patent rows", pl.read_parquet(f"{DATA}/patent_inventor_records.parquet"))]
step.append((f"filing year in {YEARS[0]}-{YEARS[1]}",
             step[-1][1].filter(pl.col("filing_year").is_between(*YEARS))))
step.append(("valid U.S. state code",
             step[-1][1].filter(pl.col("raw_state").is_in(sorted(US_STATES)))))
obs = (step[-1][1]
       .with_columns(state=pl.col("raw_state"),
                     city_norm=norm_city(pl.col("raw_city")),
                     last_norm=strip_suffix(norm_text(pl.col("raw_last"))),
                     first_full=norm_text(pl.col("raw_first")))
       .with_columns(toks=pl.col("first_full").str.split(" "))
       .with_columns(first_norm=pl.col("toks").list.get(0, null_on_oob=True),
                     mi=pl.col("toks").list.get(1, null_on_oob=True).str.slice(0, 1))
       .with_columns(first3=pl.col("first_norm").str.slice(0, 3))
       .drop("toks", "first_full"))
step.append(("non-null name & city after normalization",
             obs.drop_nulls(["city_norm", "last_norm", "first_norm"])))
obs = step[-1][1]

print("=== CLEANING FUNNEL (patent side) ===")
prev = None
for label, df in step:
    n, k = df.height, df["inventor_id"].n_unique()
    drop = f"  (-{prev - n:,} rows)" if prev is not None else ""
    print(f"  {label:<42s}: {n:>7,} rows | {k:>6,} inventors{drop}")
    prev = n
inv_cells = obs.select("inventor_id", "state", "city_norm", "last_norm").unique()
inv_variants = obs.select("inventor_id", "first_norm", "first3", "mi").unique()
inv_years = obs.select("inventor_id", "filing_year").unique()
print(f"\nname variants per inventor (spelling differs across filings): "
      f"{inv_variants.height / inv_cells['inventor_id'].n_unique():.2f}")

## 3 · Blocking and the directory side

All-pairs comparison is quadratic and infeasible (in production: 2.1M inventors ×
634M directory person-years). Instead, candidates must share a
**(state, city, surname, first3)** block — and the block's *key set is the closure
of the name relations* the ladder will use: an inventor keyed under `WIL` is also
keyed under `BIL` via the nickname map, otherwise "Bill Anderson" could never reach
`WILLIAM ANDERSON`. (Same design as the production pipeline's five name-form
channels; typos inside the first three characters still defeat the block — an
accepted, *measured* loss, as at production scale.)

The directory is scanned **lazily** and pruned with a semi-join before anything is
materialized — the small-data mirror of the production pattern (coarse `is_in`
prefilter → exact confirm) used to stream the 203 GB Data Axle and 62 GB Verisk
files. Person-level aggregation then backfills missing middle initials from other
years of the same person — production does the same within name-block groups.

In [ ]:
_n = pl.DataFrame([(k, v) for k, vs in FIRST_NICK.items() for v in vs],
                  schema=["n1", "n2"], orient="row")
NICKNAMES = pl.concat([_n, _n.select(n1="n2", n2="n1")]).unique()   # bidirectional

cells3 = inv_cells.select("state", "city_norm", "last_norm").unique()
n_total = pl.scan_parquet(f"{DATA}/directory_person_years.parquet") \
            .select(pl.len()).collect().item()
dir_py = (pl.scan_parquet(f"{DATA}/directory_person_years.parquet")
          .with_columns(city_norm=norm_city(pl.col("city")),
                        last_norm=strip_suffix(norm_text(pl.col("last_name"))),
                        first_p=norm_text(pl.col("first_name")))
          .join(cells3.lazy(), on=["state", "city_norm", "last_norm"], how="semi")
          .collect(engine="streaming"))
print(f"directory person-years: {n_total:,} scanned -> "
      f"{dir_py.height:,} at inventor blocks "
      f"({dir_py.height / n_total:.0%} survive the coarse prune)")

P = (dir_py
     .with_columns(by_dir=pl.col("year") - pl.col("age"))
     .group_by("person_id")
     .agg(pl.col("state", "city_norm", "last_norm", "first_p").first(),
          mi_p=pl.col("middle_initial").drop_nulls().first(),   # backfill across years
          years=pl.col("year").unique(),
          birth_year_dir=pl.col("by_dir").drop_nulls().median())
     .with_columns(first3=pl.col("first_p").str.slice(0, 3)))
print(f"directory persons at inventor blocks: {P.height:,} "
      f"(middle initial after backfill: "
      f"{P['mi_p'].is_not_null().sum() / P.height:.0%}, "
      f"usable birth year: {P['birth_year_dir'].is_not_null().sum() / P.height:.0%})")

# blocking keys = first3 of every observed spelling, plus nickname alternates
keys = pl.concat([
    inv_variants.select("inventor_id", key=pl.col("first3")),
    inv_variants.join(NICKNAMES, left_on="first_norm", right_on="n1")
                .select("inventor_id", key=pl.col("n2").str.slice(0, 3)),
]).unique().drop_nulls()
pairs = (inv_cells.join(keys, on="inventor_id")
         .join(P.select("person_id", "state", "city_norm", "last_norm",
                        key=pl.col("first3")),
               on=["state", "city_norm", "last_norm", "key"])
         .select("inventor_id", "person_id").unique())
cand = pairs.group_by("inventor_id").len()
print(f"candidate pairs: {pairs.height:,} for {cand.height:,} inventors "
      f"(median {cand['len'].median():.0f}, p95 {cand['len'].quantile(0.95):.0f}, "
      f"max {cand['len'].max()})")

## 4 · The matching ladder

Steps are ordered strict → lax and applied with **sequential lock-in**: a step
claims an inventor only if **exactly one** candidate satisfies it; claimed
inventors leave the pool, so a laxer step can never overturn a stricter one.
Middle initials must never *conflict* (missing on either side is consistent).
Condensed from the 10-step production ladder:

| step | rule |
|---|---|
| 1 | first name exact, MI consistent |
| 2 | one first name is a prefix of the other (`DAN` ⊂ `DANIEL`), MI consistent |
| 3 | nickname pair or misspelling (Damerau-Levenshtein ≤ 2), MI consistent |
| 4 | any step-1–3 name relation **and** a directory presence year within ±2 of a filing year |
| 5 | ambiguous pool: unique candidate with \|directory birth year − education-inferred birth year\| ≤ 2 |

Step 4 is the *timing test* (production: employment spells / residence spells must
overlap the filing window); step 5 is *age disambiguation* (production: birth year
inferred from LinkedIn education vs. directory age).

In [ ]:
from rapidfuzz import process as rf_process
from rapidfuzz.distance import DamerauLevenshtein

pv = (pairs.join(inv_variants, on="inventor_id")           # inventor spellings
      .join(P.select("person_id", "first_p", "mi_p"), on="person_id"))
pv = pv.with_columns(
    first_exact=pl.col("first_norm") == pl.col("first_p"),
    first_contained=(pl.col("first_norm").str.starts_with(pl.col("first_p"))
                     | pl.col("first_p").str.starts_with(pl.col("first_norm"))),
    mi_consistent=(pl.col("mi").is_null() | pl.col("mi_p").is_null()
                   | (pl.col("mi") == pl.col("mi_p"))),
)
pv = (pv.join(NICKNAMES.with_columns(first_nick=True),
              left_on=["first_norm", "first_p"], right_on=["n1", "n2"], how="left")
      .with_columns(first_nick=pl.col("first_nick").fill_null(False)))
uq = (pv.select("first_norm", "first_p").unique()
      .filter(pl.col("first_norm") != pl.col("first_p")))
d = rf_process.cpdist(uq["first_norm"].to_list(), uq["first_p"].to_list(),
                      scorer=DamerauLevenshtein.distance,
                      score_cutoff=DL_MAX, workers=-1)
uq = uq.with_columns(first_missp=pl.Series(np.asarray(d) <= DL_MAX))
pv = (pv.join(uq, on=["first_norm", "first_p"], how="left")
      .with_columns(first_missp=pl.col("first_missp").fill_null(False)))

timing = (pairs.join(inv_years, on="inventor_id")
          .join(P.select("person_id", "years").explode("years"), on="person_id")
          .filter((pl.col("filing_year") - pl.col("years")).abs() <= TIMING_SLACK)
          .select("inventor_id", "person_id").unique()
          .with_columns(timing_ok=True))

flags = (pv.group_by("inventor_id", "person_id")
         .agg(s1=(pl.col("first_exact") & pl.col("mi_consistent")).any(),
              s2=(pl.col("first_contained") & ~pl.col("first_exact")
                  & pl.col("mi_consistent")).any(),
              s3=((pl.col("first_nick") | pl.col("first_missp"))
                  & ~pl.col("first_exact") & pl.col("mi_consistent")).any())
         .join(timing, on=["inventor_id", "person_id"], how="left")
         .with_columns(timing_ok=pl.col("timing_ok").fill_null(False))
         .with_columns(any_name=pl.col("s1") | pl.col("s2") | pl.col("s3")))
STEP_CONDS = {1: pl.col("s1"), 2: pl.col("s2"), 3: pl.col("s3"),
              4: pl.col("any_name") & pl.col("timing_ok")}

def run_steps(flag_df, pool, steps):
    locked = []
    for s in steps:
        q = flag_df.join(pool, on="inventor_id", how="semi").filter(STEP_CONDS[s])
        u = (q.group_by("inventor_id")
             .agg(n=pl.col("person_id").n_unique(), person_id=pl.col("person_id").first())
             .filter(pl.col("n") == 1).select("inventor_id", "person_id")
             .with_columns(match_step=pl.lit(s, dtype=pl.Int8)))
        locked.append(u)
        pool = pool.join(u, on="inventor_id", how="anti")
        print(f"  step {s}: locked {u.height:>5,}   pool -> {pool.height:,}")
    return pl.concat(locked), pool

pool = obs.select("inventor_id").unique()
n_enter = pool.height
print(f"inventors entering the matcher: {n_enter:,}")
matches, pool = run_steps(flags, pool, [1, 2, 3, 4])

In [ ]:
# step 5 -- age disambiguation of the still-ambiguous pool
birth_est = pl.read_parquet(f"{DATA}/inventor_birth_est.parquet")
amb = (flags.filter(pl.col("any_name"))
       .join(pool, on="inventor_id", how="semi")
       .select("inventor_id", "person_id"))
step5 = (amb.join(birth_est, on="inventor_id")
         .join(P.select("person_id", "birth_year_dir"), on="person_id")
         .filter((pl.col("birth_year_dir") - pl.col("birth_year_est")).abs() <= AGE_TOL)
         .group_by("inventor_id")
         .agg(n=pl.col("person_id").n_unique(), person_id=pl.col("person_id").first())
         .filter(pl.col("n") == 1).select("inventor_id", "person_id")
         .with_columns(match_step=pl.lit(5, dtype=pl.Int8)))
print(f"  step 5: locked {step5.height:>5,} of the ambiguous pool by age")
matches = pl.concat([matches, step5])
pool = pool.join(step5, on="inventor_id", how="anti")

n_amb = amb.join(pool, on="inventor_id", how="semi")["inventor_id"].n_unique()
n_zero = pool.height - n_amb
funnel = pl.concat([
    matches.group_by("match_step").len().sort("match_step")
           .select(stage=pl.format("step {}", "match_step"),
                   inventors=pl.col("len").cast(pl.Int64)),
    pl.DataFrame({"stage": ["ambiguous", "no candidate"],
                  "inventors": [n_amb, n_zero]}),
]).with_columns(share=pl.col("inventors") / n_enter)
funnel.write_csv(f"{OUT}/table1_matching_funnel.csv")

print(f"\n=== MATCHING FUNNEL ===  (entering: {n_enter:,})")
for st, n, sh in funnel.iter_rows():
    print(f"  {st:<14s}: {n:>6,}  ({sh:5.1%})")
print(f"  {'MATCHED':<14s}: {matches.height:>6,}  ({matches.height / n_enter:5.1%})"
      f"   [production, real data: 72.9% vs. 69.0% in the published benchmark]")

## 5 · Validation against ground truth

Synthetic data's payoff: the matcher can be *graded*. **Precision** by ladder step
(was the matched person the right one?), **recall** (how many findable inventors
were found?), and the **false-match rate** on inventors who are not in the
directory at all — the quantity that is invisible in real data and the reason the
production pipeline leans on timing tests, residence trajectories, and audit
samples. Production has no truth table, so it validates differently: match rates
against the published benchmark, age cross-checks flagged at scale, and per-step
manual audits of LinkedIn profiles.

In [ ]:
val = (matches.join(truth, on="inventor_id", how="left")
       .with_columns(correct=pl.col("person_id") == pl.col("true_person_id")))
tab = (val.group_by("match_step").agg(matched=pl.len(), correct=pl.col("correct").sum())
       .sort("match_step")
       .with_columns(precision=pl.col("correct") / pl.col("matched")))
overall = pl.DataFrame({"match_step": [None], "matched": [val.height],
                        "correct": [val["correct"].sum()],
                        "precision": [val["correct"].sum() / val.height]})
tab = (pl.concat([tab, overall], how="vertical_relaxed")
       .with_columns(step=pl.coalesce(pl.col("match_step").cast(pl.Utf8),
                                      pl.lit("ALL")))
       .select("step", "matched", "correct", "precision"))
tab.write_csv(f"{OUT}/table2_validation.csv")
print("=== PRECISION BY LADDER STEP ===")
print(tab)

findable = truth.filter(pl.col("true_person_id").is_not_null()).height
recall = val.filter(pl.col("correct")).height / findable
out_dir = truth.filter(pl.col("true_person_id").is_null())
fm = matches.join(out_dir, on="inventor_id", how="semi").height
print(f"recall (correct / {findable:,} findable inventors): {recall:.1%}")
print(f"false-match rate on {out_dir.height} out-of-directory inventors: "
      f"{fm / out_dir.height:.1%}  <- the risk timing/trajectory tests exist to bound")

## 6 · Panel assembly, funnel figure, coverage figure

Matched inventors × all years 2006–2025 (a rectangular skeleton, as in the
production master panel), joined to directory person–years (presence,
homeownership) and to patenting activity (filings per year, first filing year,
event time). Figures: the funnel's disposition of every inventor, and coverage
over calendar time.

In [ ]:
years_df = pl.DataFrame({"year": np.arange(YEARS[0], YEARS[1] + 1)})
n_pat = obs.group_by("inventor_id", year=pl.col("filing_year")).agg(n_filed=pl.len())
first_f = obs.group_by("inventor_id").agg(first_filing=pl.col("filing_year").min())
panel = (matches.join(years_df, how="cross")
         .join(dir_py.select("person_id", "year", "homeowner", "age"),
               on=["person_id", "year"], how="left")
         .join(n_pat, on=["inventor_id", "year"], how="left")
         .join(first_f, on="inventor_id", how="left")
         .with_columns(in_directory=pl.col("homeowner").is_not_null(),
                       n_filed=pl.col("n_filed").fill_null(0),
                       event_time=pl.col("year") - pl.col("first_filing")))
panel.write_parquet(f"{OUT}/inventor_panel.parquet")
cov = (panel.group_by("year")
       .agg(in_dir=pl.col("in_directory").mean(),
            own=pl.col("homeowner").drop_nulls().mean())
       .sort("year"))
print(f"panel: {panel.height:,} inventor-years "
      f"({matches.height:,} inventors x {YEARS[1] - YEARS[0] + 1} years); "
      f"directory coverage {panel['in_directory'].mean():.1%}")

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

INK, SEC, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, BASE = "#e1e0d9", "#c3c2b7"
BLUE, ORANGE = "#2a78d6", "#eb6834"
BLUES5 = ["#86b6ef", "#5598e7", "#2a78d6", "#256abf", "#184f95"]  # ordinal ramp
matplotlib.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Segoe UI", "Arial", "DejaVu Sans"],
    "axes.edgecolor": BASE, "axes.labelcolor": SEC, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.8, "axes.spines.top": False,
    "axes.spines.right": False, "figure.dpi": 110, "savefig.dpi": 200,
    "savefig.bbox": "tight", "axes.titlesize": 12, "axes.titleweight": "bold",
})

# -- Figure 1: disposition of every inventor entering the matcher -------------
fig, ax = plt.subplots(figsize=(7.6, 3.6))
stages = funnel["stage"].to_list()[::-1]
counts = funnel["inventors"].to_list()[::-1]
colors = ([BASE, BASE] + BLUES5[::-1])[:len(stages)]   # gray for the unmatched
ax.barh(stages, counts, color=colors, height=0.62)
for y, c in enumerate(counts):
    ax.text(c + n_enter * 0.006, y, f"{c:,}  ({c / n_enter:.0%})",
            va="center", fontsize=9, color=SEC)
ax.set_xlim(0, max(counts) * 1.22)
ax.set_xlabel("inventors")
ax.set_title("Matching funnel — disposition of every inventor")
ax.grid(axis="y", visible=False)
fig.savefig(f"{OUT}/fig1_matching_funnel.png")
plt.show()

# -- Figure 2: coverage over calendar time ------------------------------------
fig, ax = plt.subplots(figsize=(7.6, 3.8))
ax.plot(cov["year"], cov["in_dir"] * 100, color=BLUE, lw=2,
        marker="o", ms=4, label="in directory")
ax.plot(cov["year"], cov["own"] * 100, color=ORANGE, lw=2,
        marker="o", ms=4, label="homeowner (if present)")
for col, colr in (("in_dir", BLUE), ("own", ORANGE)):
    ax.annotate(f"{cov[col][-1] * 100:.0f}%", (cov["year"][-1], cov[col][-1] * 100),
                textcoords="offset points", xytext=(6, -3), color=colr, fontsize=9)
ax.set_ylim(0, 100)
ax.set_xticks(np.arange(YEARS[0], YEARS[1] + 1, 3))
ax.set_ylabel("% of matched inventors")
ax.set_title("Directory coverage and homeownership by year")
ax.legend(frameon=False, loc="lower right")
fig.savefig(f"{OUT}/fig2_coverage_by_year.png")
plt.show()

## 7 · Event study — homeownership around the first patent

Linear probability model with **inventor and calendar-year fixed effects**,
standard errors clustered by inventor:

```
own_it = α_i + λ_t + Σ_{k ≠ −1} β_k · 1[t − t_i^first = k] + ε_it
```

implemented from scratch in NumPy (alternating within-transformation for the
two-way fixed effects; CR1 cluster-robust sandwich). The DGP planted a **+15 pp**
jump at k = 0 and — by construction — flat pre-trends (the age profile is linear,
so it is absorbed exactly by the two fixed effects). The estimate should land near
the truth, *mildly attenuated* by the measured false-match share, whose members
carry no true effect — the same attenuation logic that motivates the production
pipeline's audit samples.

In [ ]:
def twfe(df, ycol, xcols, unit, timec):
    """Two-way FE OLS with CR1 cluster-robust SEs (clustered on `unit`)."""
    uid = df[unit].rank("dense").to_numpy().astype(int) - 1
    tid = df[timec].rank("dense").to_numpy().astype(int) - 1
    Z = np.column_stack([df[ycol].to_numpy().astype(float),
                         df.select(xcols).to_numpy().astype(float)])
    for _ in range(60):                       # alternating demeaning
        m = np.zeros((uid.max() + 1, Z.shape[1]))
        np.add.at(m, uid, Z)
        Z -= (m / np.bincount(uid)[:, None])[uid]
        m = np.zeros((tid.max() + 1, Z.shape[1]))
        np.add.at(m, tid, Z)
        step = (m / np.bincount(tid)[:, None])[tid]
        Z -= step
        if np.abs(step).max() < 1e-10:
            break
    y, X = Z[:, 0], Z[:, 1:]
    XtX_inv = np.linalg.inv(X.T @ X)
    b = XtX_inv @ X.T @ y
    e = y - X @ b
    G = uid.max() + 1
    S = np.zeros((G, X.shape[1]))
    np.add.at(S, uid, X * e[:, None])
    n, k = X.shape
    dof = G / (G - 1) * (n - 1) / (n - k)
    V = dof * XtX_inv @ (S.T @ S) @ XtX_inv
    r2 = 1 - (e @ e) / (y @ y)
    return b, np.sqrt(np.diag(V)), n, G, r2

es = (panel.filter(pl.col("in_directory"))
      .with_columns(k=pl.col("event_time").clip(*EVENT_WIN),
                    own=pl.col("homeowner").cast(pl.Float64)))
KS = [k for k in range(EVENT_WIN[0], EVENT_WIN[1] + 1) if k != -1]
es = es.with_columns([(pl.col("k") == k).cast(pl.Float64).alias(f"D{k}")
                      for k in KS])

b_es, se_es, *_ = twfe(es, "own", [f"D{k}" for k in KS], "inventor_id", "year")
es_tab = pl.DataFrame({"event_time": KS, "coef": b_es, "se": se_es})
es_tab.write_csv(f"{OUT}/table3b_event_study.csv")

es = es.with_columns(post=(pl.col("event_time") >= 0).cast(pl.Float64))
b, se, n, G, r2 = twfe(es, "own", ["post"], "inventor_id", "year")
lo, hi = b[0] - 1.96 * se[0], b[0] + 1.96 * se[0]
print("=== TABLE 3 - Homeownership around the first patent filing (LPM) ===")
print(f"  {'':<26s}{'coef':>8s}{'(SE)':>10s}{'t':>7s}      95% CI")
print(f"  {'post (>= first filing)':<26s}{b[0]:>8.4f}{f'({se[0]:.4f})':>10s}"
      f"{b[0] / se[0]:>7.1f}   [{lo:.3f}, {hi:.3f}]")
print(f"  inventor FE + year FE; N = {n:,}; clusters = {G:,}; "
      f"within-R2 = {r2:.4f}")
print(f"  planted effect: +{TRUE_ATT:.3f}")
pl.DataFrame({"term": ["post"], "coef": [b[0]], "se": [se[0]],
              "ci_lo": [lo], "ci_hi": [hi], "N": [n], "clusters": [G],
              "within_r2": [r2]}).write_csv(f"{OUT}/table3_did.csv")

In [ ]:
# -- Figure 3: event-study coefficients ---------------------------------------
xs = np.array(KS + [-1]); order = np.argsort(xs)
ys = np.append(b_es, 0.0)[order]
ses = np.append(se_es, 0.0)[order]
xs = xs[order]

fig, ax = plt.subplots(figsize=(7.6, 4.2))
ax.axhline(0, color=BASE, lw=1)
ax.axhline(TRUE_ATT, color=SEC, lw=1, ls="--")
ax.annotate(f"planted effect (+{TRUE_ATT * 100:.0f} pp)",
            (EVENT_WIN[0], TRUE_ATT), textcoords="offset points",
            xytext=(2, 5), fontsize=9, color=SEC)
ax.axvline(-0.5, color=BASE, lw=1, ls=":")
ax.errorbar(xs, ys, yerr=1.96 * ses, fmt="o", color=BLUE, ms=5,
            lw=0, elinewidth=1.6, capsize=3, ecolor=BLUE)
ax.set_xticks(xs)
ax.set_xticklabels([rf"$\leq${EVENT_WIN[0]}" if k == EVENT_WIN[0]
                    else (rf"$\geq${EVENT_WIN[1]}" if k == EVENT_WIN[1] else str(k))
                    for k in xs])
ax.set_xlabel("years since first patent filing")
ax.set_ylabel("effect on P(homeowner), rel. to k = -1")
ax.set_title("Event study — homeownership around the first patent")
ax.grid(axis="x", visible=False)
fig.savefig(f"{OUT}/fig3_event_study.png")
plt.show()

print(f"\nnotebook total: {time.time() - t00:.1f}s")

## 8 · What changes at production scale

The logic above is the production logic; the engineering around it is what scale
changes. In the real pipeline (18 notebooks, four proprietary sources, ~7h cold /
~3min warm):

- **Nothing large is ever `collect()`ed.** The 203 GB directory and the 1.79B-row
  position table are lazily scanned with `sink_parquet`/streaming engines; the
  "coarse prefilter → exact confirm" two-stage scan used in Section 3 is the
  workhorse pattern, reused across three different sources.
- **Most-selective-filter-first.** The 4-key block is applied *before* any
  person-level aggregation, cutting ~80% of rows before the first `group_by`.
- **Every expensive scan is cached with a fingerprint** — a JSON sidecar records
  the parameters that built the cache; a mismatch triggers a rebuild instead of a
  silently stale read.
- **Every stage prints its funnel** (the Table A.1 convention used throughout this
  notebook), so sample attrition is auditable end to end.
- **Reproducibility is tested, not assumed**: a from-scratch replication on an
  independent machine reproduced the full crosswalk with a 100.0000% match-level
  agreement rate and ≥99.9995% column-level agreement in the final panel.
- **Validation without ground truth**: match rates benchmarked against the
  published replication target (72.9% achieved vs. 69.0%), age cross-checks
  flagged at scale, per-step manual audit samples, and — as in Section 5 — an
  explicit accounting of where false matches could enter and which test bounds them.